In [ ]:
import time
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from sklearn import preprocessing
import joblib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Data ─────────────────────────────────────────────────────────────────────
DATA_PATH = 'suspension_dataset.mat'   # update path if needed
N_SAMPLES = 20000   # samples per signal pair; set None for full 100k (GPU recommended)
SEQ_LEN   = 1000       # road history window [samples] = 500 ms at 1 kHz
STRIDE    = 50        # step between windows (reduces dataset size)

# ── Model ────────────────────────────────────────────────────────────────────
HIDDEN_DIM  = 128      # GRU hidden state size
FC_WIDTH    = 200     # width of each fully-connected layer in the MLP decoder
# MAX_BODY_TRAVEL and MAX_TYRE_TRAVEL are computed from the data after loading
MAX_BODY_TRAVEL = None
MAX_TYRE_TRAVEL = None
LAMBDA_PHYS = 0.5  # weight for physics loss term (0 = no physics loss, 1 = equal weight to MSE loss)

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE   = 256
EPOCHS       = 300
LR_PATIENCE  = 10     # epochs before ReduceLROnPlateau reduces LR
LR_FACTOR    = 0.9
LR_MIN       = 1e-16
MAX_PATIENCE = 50     # early stopping patience
LOG_EVERY    = 5      # print batch progress every N batches

Ts = 0.001            # sampling interval [s]
N  = 100_000          # full period length in the .mat file

print('Configuration set.')
print(f'  N_SAMPLES = {N_SAMPLES}  (None = full 100k)')
print(f'  SEQ_LEN   = {SEQ_LEN} samples ({SEQ_LEN * Ts * 1000:.0f} ms of road history)')

In [ ]:
raw     = sio.loadmat(DATA_PATH)
CLASSES = ['A', 'B', 'C', 'D', 'E']
cap     = N_SAMPLES if N_SAMPLES is not None else N

road_signals, body_signals, tyre_signals = [], [], []

for cls in CLASSES:
    road_x = raw[f'hx{cls}'].flatten()
    body_x = raw[f'bodyxsuppressed{cls}'].flatten()
    tyre_x = raw[f'tirexsuppressed{cls}'].flatten()

    # Periods 2, 3, 4 — skip period 1 (transient)
    for p in range(1, 4):
        sl = slice(p * N, p * N + cap)
        road_signals.append(road_x[:cap])
        body_signals.append(body_x[sl])
        tyre_signals.append(tyre_x[sl])

print(f'Signal pairs loaded : {len(road_signals)}  (expected 15)')
print(f'Samples per pair    : {road_signals[0].shape[0]:,}')

# ── Sanity plot ───────────────────────────────────────────────────────────────
t_ax = np.arange(cap) * Ts
fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)
for ax, sig, lbl, col in zip(
    axes,
    [road_signals[0], body_signals[0], tyre_signals[0]],
    ['Road [m]', 'Body disp. [m]', 'Tyre disp. [m]'],
    ['#0066cc', 'tomato', '#669933']
):
    ax.plot(t_ax, sig, lw=0.7, color=col)
    ax.set_ylabel(lbl)
    ax.grid(True, linestyle='--', linewidth=0.5)
axes[0].set_title(f'Class A — Unsuppressed, Period 2  ({cap:,} samples)')
axes[2].set_xlabel('Time [s]')
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Evaluate on a single road class (continuous, unshuffled, unsuppressed only)
# ═══════════════════════════════════════════════════════════════════════════════

# ── Choose signal ─────────────────────────────────────────────────────────────
ROAD_CLASS = 'A'      # 'A', 'B', 'C', 'D', 'E'
PERIOD     = 2        # 2, 3, or 4

# ── Load raw signal pair (unsuppressed only) ─────────────────────────────────
road_raw = raw[f'hx{ROAD_CLASS}'].flatten()[:cap]
body_raw = raw[f'bodyxsuppressed{ROAD_CLASS}'].flatten()[(PERIOD-1)*N : (PERIOD-1)*N + cap]
tyre_raw = raw[f'tirexsuppressed{ROAD_CLASS}'].flatten()[(PERIOD-1)*N : (PERIOD-1)*N + cap]

# ── Scale ─────────────────────────────────────────────────────────────────────
road_sc = (road_raw - road_mean) / road_std

# ── Build continuous windows (stride=1, no shuffle) ───────────────────────────
windows = np.lib.stride_tricks.sliding_window_view(road_sc, SEQ_LEN)
n = min(len(windows), len(body_raw) - SEQ_LEN + 1)

X_inf = torch.tensor(windows[:n], dtype=torch.float32).unsqueeze(-1).to(device)

# ── Inference in batches ──────────────────────────────────────────────────────
model.load_state_dict(torch.load('best_suspension_gru.pt', map_location=device))
model.eval()

preds_list = []
with torch.no_grad():
    for i in range(0, n, BATCH_SIZE):
        preds_list.append(model(X_inf[i:i+BATCH_SIZE]).cpu().numpy())

preds_sc = np.concatenate(preds_list, axis=0)  # (n, 2) scaled

# ── Inverse-transform to metres ──────────────────────────────────────────────
preds_body = preds_sc[:, 0] * body_std + body_mean
preds_tyre = preds_sc[:, 1] * tyre_std + tyre_mean
targets_body = body_raw[SEQ_LEN - 1 : SEQ_LEN - 1 + n]
targets_tyre = tyre_raw[SEQ_LEN - 1 : SEQ_LEN - 1 + n]

# ── Metrics ───────────────────────────────────────────────────────────────────
title = f'Class {ROAD_CLASS} — Unsuppressed — Period {PERIOD}'

def report(pred, true, label):
    rmse = np.sqrt(np.mean((pred - true) ** 2))
    mae  = np.mean(np.abs(pred - true))
    print(f'  {label:6s}  RMSE: {rmse*1000:.4f} mm   MAE: {mae*1000:.4f} mm')

print(f'{title}')
print(f'Samples: {n:,}\n')
report(preds_body, targets_body, 'Body')
report(preds_tyre, targets_tyre, 'Tyre')